# Fine-mapping benchmark — interactive results explorer

An interactive version of the by-split report. **Edit the CONFIG cell, then Run All.**

You specify:
- **`INCLUDE`** — scenario settings to keep (e.g. `S = 1`, or `list(model="sparse", enrichment="none")`).
- **`EXCLUDE`** — settings to drop (e.g. the unrealistic `phi = 0.0075`).
- **`BOUNDARIES`** — variables never pooled across; you get one set of graphs per combination of their levels (default `model` × `ld`).

For every split, and for each of **AUPRC / FDR control / PIP calibration**, you get a graph showing how the metric
changes with respect to *every other variable that still varies* — one panel per such variable, one line per method,
aggregating over the remaining free variables. Nothing is combined across the boundary variables.

Variables you can reference: `model`, `ld` (in-sample / ref500 / ref200), `S`, `phi`, `region_length`,
`p_causal`, `enrichment` (none / 2.7 / 5.4 / 8.1 / 10.8), `annotation_type` (none / binary / continuous).

> Requires an R (IRkernel) kernel. If you don't have it: in R run `install.packages("IRkernel"); IRkernel::installspec()`.
> AUPRC uses the corrected per-replicate (macro) values.

In [ ]:
# ============================ EDIT THIS CELL ============================
DATA <- "/Users/ls1020/Documents/StatML/fine-mapping-benchmark/results/iter002_fixed/combined_scenario_metrics_macroAP.rds"

# Keep ONLY these values (omit a variable to keep ALL its levels).
#   e.g. INCLUDE <- list(S = 1, model = "sparse", enrichment = "none")
#        INCLUDE <- list(phi = c(0.1, 0.2, 0.4))     # a subset keeps the variable as an axis
INCLUDE <- list()

# Drop these values.
#   e.g. EXCLUDE <- list(phi = c(0.0075, 0.05))
EXCLUDE <- list(phi = 0.0075)

# NEVER pool across these -> one set of graphs per combination of their levels.
BOUNDARIES <- c("model", "ld")

AGG          <- median                 # across-scenario aggregator: median or mean
DROP_METHODS <- c("polyfun_oracle")     # oracle is a ceiling, not a real method
# =======================================================================

In [ ]:
# ---- setup: libraries, data, canonical variables, helpers (run once) -------
suppressPackageStartupMessages({library(ggplot2); library(scales)})

s <- readRDS(DATA)
num <- function(x) as.numeric(as.character(x))
s$S <- num(s$S); s$phi <- num(s$phi); s$region_length <- num(s$region_size)
s$ld <- ifelse(is.na(s$n_ref), "in-sample", paste0("ref", s$n_ref))
s$enrichment <- ifelse(is.na(s$enrichment_fold), "none", as.character(s$enrichment_fold))
s$p_causal   <- ifelse(is.na(s$p_causal), "none(sparse)", as.character(s$p_causal))

VARS <- c("model","ld","S","phi","region_length","p_causal","enrichment","annotation_type")
METRICS <- list(
  AUPRC       = list(col="ap",                    dir="higher", hline=NA,   lim=c(0,1)),
  FDR         = list(col="max_fdr_violation_n20", dir="lower",  hline=0.05, lim=c(0,1)),
  Calibration = list(col="hi_pip_reliab",         dir="higher", hline=0.9,  lim=c(0,1)))
LABS <- c(model="model", ld="LD regime", S="number of causal variants S",
  phi="per-variant heritability \u03c6", region_length="region length (variants)",
  p_causal="p_causal (sparse_inf mix)", enrichment="annotation enrichment",
  annotation_type="annotation type")
ORD <- list(enrichment=c("none","2.7","5.4","8.1","10.8"),
  p_causal=c("none(sparse)","0.5","0.7","0.9","1"),
  annotation_type=c("none","binary","continuous"),
  ld=c("in-sample","ref500","ref200"), model=c("sparse","sparse_inf"))
AGG_LABEL <- if (identical(AGG, median)) "median" else if (identical(AGG, mean)) "mean" else "aggregate"

lvorder <- function(v, x){ u <- unique(as.character(x))
  if (!is.null(ORD[[v]])) ORD[[v]][ORD[[v]] %in% u]
  else if (suppressWarnings(!any(is.na(as.numeric(u))))) u[order(as.numeric(u))] else sort(u) }

PAL <- c(abf="#2a78d6",beatrice="#e34948",finemap="#006300",functional_beatrice="#eb6834",
  funmap="#9467bd",marginal_z="#8c8c8c",paintor="#c9a227",polyfun_est="#1baf7a",
  polyfun_ldsc="#17becf",sbayesrc="#d6277a",sparsepro="#4a3aa7",susie="#111111",susie_inf="#7fb800")
SURF <- "#fcfcfb"
th <- theme_minimal(base_size=10)+theme(
  plot.background=element_rect(fill=SURF,colour=NA), panel.background=element_rect(fill=SURF,colour=NA),
  panel.grid.minor=element_blank(), plot.title=element_text(face="bold",size=12),
  plot.subtitle=element_text(colour="#52514e",size=8.5), strip.text=element_text(face="bold",size=9),
  legend.position="bottom", legend.title=element_blank(), legend.text=element_text(size=8),
  panel.spacing=unit(8,"pt"))

freevars_of <- function(d) VARS[sapply(VARS, function(v)
  !(v %in% BOUNDARIES) && length(unique(d[[v]])) > 1L)]

subset_split <- function(sp){ d <- s
  for (b in BOUNDARIES) d <- d[as.character(d[[b]]) == as.character(sp[[b]]), ]; d }
split_label <- function(sp) paste(sapply(BOUNDARIES, function(b) sprintf("%s = %s", b, sp[[b]])), collapse=",  ")

metric_graph <- function(d, metric, lab){
  M <- METRICS[[metric]]; fv <- freevars_of(d)
  if (!length(fv)) return(NULL)
  L <- do.call(rbind, lapply(fv, function(v){
    ag <- aggregate(d[[M$col]], by=list(method=d$method, lvl=as.character(d[[v]])),
                    FUN=function(x) AGG(x, na.rm=TRUE)); names(ag)[3] <- "y"
    ag$panel <- factor(LABS[[v]], levels=unname(LABS[fv]))
    ag$xf <- factor(paste0(v,":",ag$lvl), levels=paste0(v,":",lvorder(v, d[[v]]))); ag }))
  L$method <- factor(L$method, levels=names(PAL))
  g <- ggplot(L, aes(xf, y, colour=method, group=method))
  if (!is.na(M$hline)) g <- g + geom_hline(yintercept=M$hline, linetype="22", colour="#52514e", linewidth=.3)
  g + geom_line(linewidth=.55, na.rm=TRUE) + geom_point(size=1.4, na.rm=TRUE) +
    facet_wrap(~panel, nrow=1, scales="free_x") +
    scale_x_discrete(labels=function(x) sub("^[^:]+:", "", x)) +
    scale_colour_manual(values=PAL, drop=FALSE) + coord_cartesian(ylim=M$lim) +
    guides(colour=guide_legend(nrow=2)) +
    labs(title=sprintf("%s  (%s is better)  \u2014  %s", metric,
           ifelse(M$dir=="higher","higher","lower"), lab),
         subtitle=sprintf("%s across the selected scenarios; one line per method. Each panel varies one axis and aggregates over the other free axes.", AGG_LABEL),
         x=NULL, y=metric) + th
}
cat("setup complete.\n")

In [ ]:
# ---- apply filters and show the plan ---------------------------------------
s <- s[!s$method %in% DROP_METHODS, ]
for (v in names(INCLUDE)) s <- s[as.character(s[[v]]) %in% as.character(INCLUDE[[v]]), ]
for (v in names(EXCLUDE)) s <- s[!as.character(s[[v]]) %in% as.character(EXCLUDE[[v]]), ]
stopifnot(nrow(s) > 0)
s$scen <- paste(s$job_dir, s$S, s$phi, s$region_size, sep="|")

bk <- do.call(paste, c(s[BOUNDARIES], sep=" | "))
SPLITS <- unique(s[!duplicated(bk), BOUNDARIES, drop=FALSE])
cat(sprintf("Kept %s scenario-cells (%s distinct scenarios), %d methods.\n",
    format(nrow(s), big.mark=","), format(length(unique(s$scen)), big.mark=","), length(unique(s$method))))
if (length(INCLUDE)) cat("INCLUDE:", paste(sprintf("%s=%s", names(INCLUDE), sapply(INCLUDE, paste, collapse="/")), collapse=", "), "\n")
if (length(EXCLUDE)) cat("EXCLUDE:", paste(sprintf("%s=%s", names(EXCLUDE), sapply(EXCLUDE, paste, collapse="/")), collapse=", "), "\n")
cat(sprintf("\nBoundaries {%s} -> %d split(s). Free variables per split:\n", paste(BOUNDARIES, collapse=", "), nrow(SPLITS)))
for (i in seq_len(nrow(SPLITS))) {
  d <- subset_split(SPLITS[i,,drop=FALSE])
  cat(sprintf("  [%d] %-40s  free: %s\n", i, split_label(SPLITS[i,,drop=FALSE]),
      paste(freevars_of(d), collapse=", ")))
}

In [ ]:
# ---- render: for each split, the three metric graphs -----------------------
for (i in seq_len(nrow(SPLITS))) {
  sp <- SPLITS[i,,drop=FALSE]; d <- subset_split(sp); lab <- split_label(sp)
  fv <- freevars_of(d)
  cat(sprintf("\n=============== SPLIT %d/%d : %s  (n=%d scenarios) ===============\n",
      i, nrow(SPLITS), lab, length(unique(d$scen))))
  if (!length(fv)) { cat("  (no free variables — every non-boundary axis is fixed; see the summary table below)\n"); next }
  w <- max(6, 2 + 2.3 * length(fv))
  for (mn in names(METRICS)) {
    options(repr.plot.width = w, repr.plot.height = 3.7)
    print(metric_graph(d, mn, lab))
  }
}

In [ ]:
# ---- numeric summary tables -------------------------------------------------
# summary_table(metric, variable) : per split, the aggregated metric by method x that variable's levels.
#   e.g. summary_table("AUPRC", "enrichment") ; summary_table("FDR", "S")
summary_table <- function(metric, variable){
  M <- METRICS[[metric]]
  for (i in seq_len(nrow(SPLITS))) {
    sp <- SPLITS[i,,drop=FALSE]; d <- subset_split(sp)
    if (length(unique(d[[variable]])) < 2L) next
    ag <- aggregate(d[[M$col]], by=list(method=d$method, lvl=as.character(d[[variable]])),
                    FUN=function(x) round(AGG(x, na.rm=TRUE), 3))
    tab <- reshape(ag, idvar="method", timevar="lvl", direction="wide")
    names(tab) <- sub("^x\\.", "", names(tab))
    ord <- c("method", lvorder(variable, d[[variable]]))
    tab <- tab[, intersect(ord, names(tab))]
    cat(sprintf("\n%s  by  %s   (%s)   [%s]\n", metric, variable, split_label(sp), AGG_LABEL))
    print(tab, row.names = FALSE)
  }
}
summary_table("AUPRC", freevars_of(subset_split(SPLITS[1,,drop=FALSE]))[1])

### Notes

- **Boundaries are never pooled.** Add to `BOUNDARIES` (e.g. `c("model","ld","annotation_type")`) to also keep annotation types separate — recommended if you compare across them, since binary and continuous behave differently.
- **"Aggregates over the other free axes."** In each panel, the x-axis is one variable and every *other* still-varying variable is collapsed with the chosen aggregator (`median` by default). Fix a variable via `INCLUDE` to remove it as an axis.
- **AUPRC** is the corrected per-replicate (macro) average. **FDR** is the guarded max violation of the FDR ≤ 1−t bound (dashed line = 0.05; lower is better). **Calibration** is the fraction of PIP ≥ 0.9 calls that are truly causal (dashed line = 0.9; higher is better; a gap means no such calls).
- `enrichment = none` is the unannotated arm (there `annotation_type = none` and functional/plain BEATRICE coincide).